In [ ]:

"""
DOI GNSS Station List → ArcGIS Online Feature Layer
-----------------------------------------------------
Reads an updatable .csv from Azure Blob Storage,
converts ECEF XYZ → WGS84 lat/lon, and publishes/overwrites
a Hosted Feature Layer on AGOL.

Run inside ArcGIS Pro Python Notebook — paste each cell block
into a separate cell and run top to bottom.
"""

# Cell 1: Imports
import math
import requests
import pandas as pd
from io import StringIO
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
from arcgis.geometry import Point, Polygon
from arcgis.features import GeoAccessor, GeoSeriesAccessor

print("Imports loaded.")

In [ ]:
#Cell 2: Configuration
BLOB_URL = "https://YOUR_BLOB_STORAGE_ACCOUNT.blob.core.windows.net/YOUR_BLOB_CONTAINER/StationInformation/StationList_Latest.csv"

# Your current GNSS station feature layer
FEATURE_LAYER_ITEM_ID = "<LEAVE BLANK TO CREATE NEW FEATURE SERVICE> <ENTER FEATURE ID TO OVERWRITE WITH UPDATE"

LAYER_TITLE = "DOI GNSS Stations"
LAYER_TAGS  = ["GNSS", "DOI", "GPS", "stations"]

# Tectonic plate → reference frame mapping
PLATE_TO_REF = {
    "North America": "NAD83(2011)",
    "Pacific":       "NAD83(PA11)",
    "Mariana":       "NAD83(MA11)",
    "Caribbean":     "NAD83(2011)",
}

print("Config ready.")

In [ ]:
# Cell 3: ECEF → WGS84 conversion
def ecef_to_latlon(x, y, z):
    a  = 6378137.0
    e2 = 0.00669437999014
    lon = math.degrees(math.atan2(y, x))
    lat = math.atan2(z, math.sqrt(x**2 + y**2))
    for _ in range(10):
        sin_lat = math.sin(lat)
        N   = a / math.sqrt(1 - e2 * sin_lat**2)
        lat = math.atan2(z + e2 * N * sin_lat, math.sqrt(x**2 + y**2))
    return math.degrees(lat), lon

print("ECEF conversion ready.")


In [ ]:
# Cell 4: Fetch & parse latest CSV
def fetch_station_data():
    print(f"Fetching: {BLOB_URL}")
    r = requests.get(BLOB_URL, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), encoding_errors="ignore")
    df.columns = [c.encode("ascii", "ignore").decode().strip().strip('"') for c in df.columns]
    print("Raw columns:", df.columns.tolist())
    for col in ["m_OriginalPosition_m_X", "m_OriginalPosition_m_Y", "m_OriginalPosition_m_Z"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["m_OriginalPosition_m_X", "m_OriginalPosition_m_Y", "m_OriginalPosition_m_Z"])
    latlon = df.apply(lambda row: ecef_to_latlon(row.m_OriginalPosition_m_X, row.m_OriginalPosition_m_Y, row.m_OriginalPosition_m_Z), axis=1, result_type="expand")
    df["latitude"]  = latlon[0]
    df["longitude"] = latlon[1]
    df = df[(df.latitude.between(-90, 90)) & (df.longitude.between(-180, 180))]
    df["reference_frame"] = df["m_OriginalPosition_m_TectonicPlate"].map(PLATE_TO_REF).fillna("Unknown")
    df["operational_status"] = "Online"
    df["display_status"] = df["reference_frame"] + " - Online"
    print("Final columns:", df.columns.tolist())
    print(f"  → {len(df)} stations parsed successfully")
    return df

df = fetch_station_data()
df.head()

In [ ]:
# Cell 5: NPS Park Boundary Spatial Join
def spatial_join_parks(df):
    print("Running park boundary spatial join...")
    
    # Connect to park boundary layer anonymously
    public_gis = GIS()
    item = public_gis.content.get("YOUR BOUNDARY FEATURE ID")
    park_layer = item.layers[2]
    
    # Query all park boundaries
    parks = park_layer.query(out_fields="UNIT_CODE,UNIT_NAME", return_geometry=True)
    print(f"  → {len(parks.features)} park boundaries loaded")
    
    # Initialize columns
    df["UNIT_CODE"] = None
    df["UNIT_NAME"] = None
    
    # For each station check if it falls within a park boundary
    for idx, row in df.iterrows():
        pt = Point({"x": row.longitude, "y": row.latitude, "spatialReference": {"wkid": 4326}})
        for park in parks.features:
            if park.geometry:
                poly = Polygon(park.geometry)
                if poly.contains(pt):
                    df.at[idx, "UNIT_CODE"] = park.attributes.get("UNIT_CODE")
                    df.at[idx, "UNIT_NAME"] = park.attributes.get("UNIT_NAME")
                    break
    
    matched = df["UNIT_CODE"].notna().sum()
    print(f"  → {matched} stations matched to a park boundary")
    return df

df = spatial_join_parks(df)
df.head()

In [ ]:
# Cell 6: Publish or smart-update
def build_feature(row, existing_fields=None):
    attrs = row.drop(["latitude", "longitude"]).to_dict()
    attrs = {k: (None if pd.isna(v) else v) for k, v in attrs.items()}
    if existing_fields:
        attrs = {k: v for k, v in attrs.items() if k in existing_fields}
    return {
        "geometry": {"x": row.longitude, "y": row.latitude, "spatialReference": {"wkid": 4326}},
        "attributes": attrs,
    }

def publish_or_update(gis, df):
    if FEATURE_LAYER_ITEM_ID:
        print(f"Updating existing layer: {FEATURE_LAYER_ITEM_ID}")
        item  = gis.content.get(FEATURE_LAYER_ITEM_ID)
        flc   = FeatureLayerCollection.fromitem(item)
        layer = flc.layers[0]
        existing_fields = {f.name for f in layer.properties.fields}
        existing = layer.query(out_fields="m_StationCode,OBJECTID,operational_status").features
        existing_codes = {f.attributes["m_StationCode"]: f.attributes["OBJECTID"]
                         for f in existing if f.attributes.get("m_StationCode")}
        new_codes = set(df["m_StationCode"].dropna())
        adds, updates = [], []
        for _, row in df.iterrows():
            code = row.get("m_StationCode")
            feat = build_feature(row, existing_fields)
            if code in existing_codes:
                feat["attributes"]["OBJECTID"] = existing_codes[code]
                updates.append(feat)
            else:
                adds.append(feat)
        retired_oids = []
        for f in existing:
            code = f.attributes.get("m_StationCode")
            if code and code not in new_codes:
                retired_oids.append({
                    "attributes": {
                        "OBJECTID": f.attributes["OBJECTID"],
                        "operational_status": "No Longer in Service",
                        "display_status": "No Longer in Service"
                    }
                })
        result = layer.edit_features(
            adds=adds if adds else None,
            updates=updates if updates else None,
        )
        if retired_oids:
            layer.edit_features(updates=retired_oids)
        print(f"  → {len(updates)} stations updated")
        print(f"  → {len(adds)} new stations added")
        print(f"  → {len(retired_oids)} stations marked No Longer in Service")
        print("Done!")
    else:
        print("No Item ID set — creating new layer...")
        from arcgis.geometry import Point
        from arcgis.features import GeoAccessor, GeoSeriesAccessor
        sdf = df.copy()
        sdf["SHAPE"] = df.apply(lambda row: Point({"x": row.longitude, "y": row.latitude, "spatialReference": {"wkid": 4326}}), axis=1)
        item = sdf.spatial.to_featurelayer(title=LAYER_TITLE, gis=gis, tags=LAYER_TAGS)
        print(f"\n  Layer created!")
        print(f"  Item ID : {item.id}")
        print(f"  Title   : {item.title}")
        print(f"\n  *** Paste Item ID into FEATURE_LAYER_ITEM_ID in Cell 2 ***")

gis = GIS("home")
print(f"Signed in as: {gis.properties.user.username}")
publish_or_update(gis, df)